In [ ]:
# Install required dependencies
!pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")

ModuleNotFoundError: No module named 'sklearn'

## Phase 1: Data Acquisition and Preprocessing
We begin by understanding our data shape, locating missing values, handling extreme outliers, and creating categorical proxy variables.

In [ ]:
# 1. Load Data
data = pd.read_csv("../data/raw/data.csv")
print(f"Raw Data Shape: {data.shape}\n")

columns_to_keep = ['Age', 'Gender', 'StudyHours', 'Attendance', 'AssignmentCompletion', 'ExamScore', 'FinalGrade']
df = data[columns_to_keep].copy()

# 2. Missing Values Analysis
missing_data = df.isnull().sum()
print("Missing Values:\n", missing_data[missing_data > 0])
df = df.dropna()

# 3. Outlier Detection (Tukey's IQR Method on ExamScore)
Q1 = df['ExamScore'].quantile(0.25)
Q3 = df['ExamScore'].quantile(0.75)
IQR = Q3 - Q1
lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df['ExamScore'] < lower_bound) | (df['ExamScore'] > upper_bound)]
print(f"\nDetected {len(outliers)} outliers in ExamScore. Removing them for robust analysis.")
df = df[(df['ExamScore'] >= lower_bound) & (df['ExamScore'] <= upper_bound)].copy()

# 4. Feature Engineering: Self-Assessment Categorization
df['SelfAssessment_Level'] = pd.cut(
    df['AssignmentCompletion'], 
    bins=[-np.inf, 49, 79, np.inf], 
    labels=['Low', 'Moderate', 'High'],
    ordered=True
)
print("\nCleaned Data Sample:")
display(df.head())

NameError: name 'pd' is not defined

## Phase 2: Exploratory Data Analysis (EDA)
A structured approach to uncovering the fundamental behaviors of our data.

### 2.1 Univariate Analysis
Examining individual variables in isolation to understand their central tendency, dispersion, and overall distribution shapes.

In [ ]:
# Descriptive statistical summary table
display(df[['AssignmentCompletion', 'ExamScore', 'StudyHours', 'Attendance']].describe().T)

# Plotting Univariate Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['ExamScore'], kde=True, bins=30, color='mediumpurple', ax=axes[0])
axes[0].set_title("Distribution of Exam Scores (Outcome)", weight='bold')
axes[0].set_xlabel("Exam Score")

sns.histplot(df['AssignmentCompletion'], kde=True, bins=30, color='darkcyan', ax=axes[1])
axes[1].set_title("Distribution of Assignment Completion (Predictor)", weight='bold')
axes[1].set_xlabel("Assignment Completion (%)")

sns.countplot(x='SelfAssessment_Level', data=df, palette="viridis", ax=axes[2])
axes[2].set_title("Frequency of Self-Assessment Levels", weight='bold')
axes[2].set_xlabel("Categorized Level")

plt.tight_layout()
plt.show()

### 2.2 Bivariate Analysis
Analyzing the relationship exactly two variables at a time to determine empirical associations. This is where we first test our core analytical statement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Scatterplot: Continuous Predictor vs Continuous Outcome
sns.regplot(x='AssignmentCompletion', y='ExamScore', data=df, 
            scatter_kws={'alpha':0.4, 'color':'teal'}, line_kws={'color':'darkorange'}, ax=axes[0])
axes[0].set_title("Bivariate: Assignment Completion vs Exam Score", fontsize=14, weight='bold')
axes[0].set_ylabel("Exam Score")
axes[0].set_xlabel("Assignment Completion (%)")

# 2. Violin Plot: Categorical Predictor vs Continuous Outcome
sns.violinplot(x='SelfAssessment_Level', y='ExamScore', data=df, palette="Set2", inner="quartile", ax=axes[1])
axes[1].set_title("Bivariate: Exam Score Density by Level", fontsize=14, weight='bold')
axes[1].set_xlabel("Self-Assessment Level (Proxy)", fontsize=12)

plt.tight_layout()
plt.show()

### 2.3 Multivariate Analysis
Exploring relationships across more than two variables simultaneously to check for confounding variables, multicollinearity, and complex demographic interactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Correlation Heatmap for Multiple Continuous Variables
corr = df[['StudyHours', 'Attendance', 'AssignmentCompletion', 'ExamScore', 'FinalGrade']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", vmax=1, vmin=-1, center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8}, ax=axes[0])
axes[0].set_title("Multivariate: Global Correlation Heatmap", fontsize=14, weight='bold')

# 2. Scatterplot with Confidence Intervals highlighting Gender differences (3 Variables)
sns.scatterplot(x='AssignmentCompletion', y='ExamScore', hue='Gender', data=df, alpha=0.5, 
                palette="deep", ax=axes[1])
sns.regplot(x='AssignmentCompletion', y='ExamScore', data=df, scatter=False, color="black", ax=axes[1])
axes[1].set_title("Multivariate: Effect of Gender on Correlation", fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

## Phase 3: Inferential Statistics
We must mathematically prove the significance of our findings.

**Hypothesis Testing:**
- $H_0$: $\mu_{Low} = \mu_{Moderate} = \mu_{High}$ (No difference in mean exam scores across self-assessment levels)
- $H_1$: At least one group mean is significantly different.

We will first test **ANOVA assumptions** visually and statistically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Assumption 1: Normality (Q-Q Plot)
stats.probplot(df['ExamScore'], dist="norm", plot=axes[0])
axes[0].set_title("Q-Q Plot (Testing Normality of Exam Scores)", weight='bold')

# Assumption 2: Homogeneity of Variance (Residuals vs Groups)
sns.boxplot(x='SelfAssessment_Level', y='ExamScore', data=df, ax=axes[1], palette="pastel")
axes[1].set_title("Variance Spread Check across Groups", weight='bold')

plt.tight_layout()
plt.show()

# Statistical tests for assumptions
sample_data = df.sample(4000, random_state=42) if len(df) > 5000 else df
_, shapiro_p = stats.shapiro(sample_data['ExamScore'])
print(f"Shapiro-Wilk Test p-value: {shapiro_p:.4e} (If < 0.05, data is strictly non-normal, but ANOVA is robust to this for large n)")

groups = [group['ExamScore'].values for name, group in df.groupby('SelfAssessment_Level', observed=False)]
_, levene_p = stats.levene(*groups)
print(f"Levene's Test p-value: {levene_p:.4e} (If < 0.05, variances are unequal)")

In [ ]:
# Execute ANOVA
f_stat, anova_p = stats.f_oneway(*groups)
print(f"\nOne-Way ANOVA Results:")
print(f"F-Statistic: {f_stat:.2f}")
print(f"P-Value: {anova_p:.4e}")

if anova_p < 0.05:
    print("\n=> Reject the Null Hypothesis. There is a statistically significant difference.")
    print("Running Tukey HSD Post-Hoc Test to find exactly WHICH groups differ...\n")
    tukey = pairwise_tukeyhsd(endog=df['ExamScore'], groups=df['SelfAssessment_Level'], alpha=0.05)
    display(tukey.summary())
else:
    print("\n=> Fail to reject the Null Hypothesis. No significant difference found.")

## Phase 4: Predictive Modeling
Finally, we train a multiple linear regression model. We use a **Train/Test split (80/20)** to evaluate how well 'AssignmentCompletion' alongside actual control variables ('StudyHours', 'Attendance') can predict unseen 'ExamScore' data.

We also check for **Multicollinearity** using the Variance Inflation Factor (VIF).

In [ ]:
# Multicollinearity Check (VIF)
X_vif = sm.add_constant(df[['AssignmentCompletion', 'StudyHours', 'Attendance']])
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]
print("Variance Inflation Factor (VIF):")
display(vif_data)
print("Note: VIF > 5 indicates problematic multicollinearity.\n")

# Train-Test Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Fit Model on Training Data
model = smf.ols('ExamScore ~ AssignmentCompletion + StudyHours + Attendance', data=train_df).fit()
display(model.summary())

# Predict on Test Data
test_df['Predicted_ExamScore'] = model.predict(test_df)
mse = np.mean((test_df['ExamScore'] - test_df['Predicted_ExamScore'])**2)
rmse = np.sqrt(mse)
print(f"\nTest Root Mean Squared Error (RMSE): {rmse:.2f} marks")

In [ ]:
# Model Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Actual vs Predicted
sns.scatterplot(x='ExamScore', y='Predicted_ExamScore', data=test_df, alpha=0.5, color='purple', ax=axes[0])
axes[0].plot([test_df['ExamScore'].min(), test_df['ExamScore'].max()], 
             [test_df['ExamScore'].min(), test_df['ExamScore'].max()], 
             color='red', linestyle='--', linewidth=2)
axes[0].set_title("Actual vs. Predicted Exam Scores (Test Set)", weight='bold')
axes[0].set_xlabel("Actual Exam Score")
axes[0].set_ylabel("Predicted Exam Score")

# 2. Residual Distribution
residuals = model.resid
sns.histplot(residuals, bins=30, kde=True, color='teal', ax=axes[1])
axes[1].set_title("Distribution of Regression Residuals", weight='bold')
axes[1].set_xlabel("Residual (Actual - Predicted)")

plt.tight_layout()
plt.show()